In [5]:
import json, sys, copy, os, igl
import numpy as np
import meshio

def is_inside(v, f, p):
    winds = igl.winding_number(v, f, p)
    mask = np.array(np.rint(winds), dtype=int)
    
    return np.array([e % 2 == 1 for e in mask])

def change_file_extension(filename, new_extension):
  base, ext = os.path.splitext(filename)
  return base + "." + new_extension if ext else filename + "." + new_extension

In [6]:
input_surface = "/home/venky/graphics/eFlesh-Thumb.stl"
stitch_cells_cli = "/home/venky/graphics/microstructure_inflators/build/isosurface_inflator/stitch_cells_cli"
only_cube_cells = True

In [7]:
E = 0.02
nu = 0.09
cell_size = 6
out_path = str(E) + "_" + str(cell_size) + ".obj"

m = meshio.read(input_surface)
input_surface = change_file_extension(input_surface, 'obj')
m.write(input_surface)
v = m.points.astype(float)
f = m.cells[0].data
bbox = [np.amin(v, axis=0), np.amax(v, axis=0)]

corner0 = list(map(int, np.ceil(bbox[0] / cell_size)))
corner1 = list(map(int, np.floor(bbox[1] / cell_size)))

print(corner0, corner1)

def young(i, j, k):
    return E

[-7, -1, 1] [-4, 1, 3]


In [8]:
sys.path.insert(0, '/home/venky/graphics/matopt/tools/material2geometry')
from material2geometry import Material2Geometry

In [9]:
mat2geo = Material2Geometry(in_path="/home/venky/graphics/matopt/tools/material2geometry/0646_geo_1_coeffs.txt")

Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]
Res: [0.038461538461538464, 0.038461538461538464]
Width: [13, 13]


In [10]:
patterns = []
entry = {"params": [],
"symmetry": "Cubic",
"pattern": "/home/venky/graphics/microstructure_inflators/data/patterns/3D/reference_wires/pattern0646.wire",
"index": [0,0,0]}

for i in range(corner0[0], 1 + corner1[0]):
    for j in range(corner0[1], 1 + corner1[1]):
        for k in range(corner0[2], 1 + corner1[2]):
            geo_params = mat2geo.evaluate(nu, young(i, j, k))
            entry["params"] = geo_params
            entry["index"] = [i,j,k]

            samples = np.array([
                [i, j, k],
                [i+1, j, k],
                [i, j+1, k],
                [i, j, k+1],
                [i+1, j+1, k],
                [i, j+1, k+1],
                [i+1, j, k+1],
                [i+1, j+1, k+1]
            ], dtype=float)
            samples *= cell_size
            if np.all(is_inside(v, f, samples)):
                patterns.append(copy.deepcopy(entry))

with open("data.json", 'w') as f:
    json.dump(patterns, f)

In [11]:
os.system(stitch_cells_cli + " -p data.json " +
           (("--surface " + input_surface) if not only_cube_cells else "") + 
           " --gridSize " + str(cell_size) + " -o " + out_path + " -r 30")

Checking printability
Inflation graph generated
Saved space 46.9037 %
Completed 12.5 %
Checking printability
Inflation graph generated
Saved space 46.9037 %
Completed 25 %
Checking printability
Inflation graph generated
Saved space 46.9037 %
Completed 37.5 %
Checking printability
Inflation graph generated
Saved space 46.9037 %
Completed 50 %
Checking printability
Inflation graph generated
Saved space 46.9037 %
Completed 62.5 %
Checking printability
Inflation graph generated
Saved space 46.9037 %
Completed 75 %
Checking printability
Inflation graph generated
Saved space 46.9037 %
Completed 87.5 %
Checking printability
Inflation graph generated
Saved space 46.9037 %
Completed 100 %


0